In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression,LogisticRegression,Lasso,Ridge
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                              classification_report, roc_auc_score,
                              confusion_matrix, log_loss)
from sklearn.datasets import make_classification,make_regression
import warnings; warnings.filterwarnings("ignore")

In [2]:
np.random.seed(42)

In [3]:
n=2000
df=pd.DataFrame({
    "area_sqft":np.random.randint(800,4000,n),
    "bedrooms":np.random.randint(1,6,n),
    "bathrooms":np.random.randint(1,4,n),
    "age_years":np.random.randint(0,30,n),
    "floor":np.random.randint(0,20,n),
    "has_gym":np.random.randint(0,2,n),
    "has_pool":np.random.randint(0,2,n),
    "locality_score":np.random.randint(1,10,n).round(1),
})

df["price_lakh"] = (
    0.04 * df["area_sqft"]
    + 5.0 * df["bedrooms"]
    + 3.0 * df["bathrooms"]
    - 0.5 * df["age_years"]
    + 0.3 * df["floor"]
    + 8.0 * df["has_gym"]
    + 12.0 * df["has_pool"]
    + 4.0 * df["locality_score"]
    + np.random.normal(0, 15, n)  # noise
).round(2)

feature_cols=["area_sqft","bedrooms","bathrooms","age_years","floor","has_gym","has_pool","locality_score"]


In [4]:
X=df[feature_cols]
y=df["price_lakh"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

### Baseline: always predict the mean

In [5]:
baseline_pred=np.full(len(y_test),y_train.mean())
baseline_rmse=np.sqrt(mean_squared_error(y_test,baseline_pred))
print(f"Baseline(predict mean) RMSE:₹{baseline_rmse:.1f}L")

Baseline(predict mean) RMSE:₹43.5L


## LINEAR REGRESSION

In [6]:
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)

lr_rmse=np.sqrt(mean_squared_error(y_test,y_pred))
lr_r2=r2_score(y_test,y_pred)
print(f"Linear Regression RMSE:₹{lr_rmse:.1f}L R^2:{lr_r2:.1f}")

Linear Regression RMSE:₹15.1L R^2:0.9


In [7]:
coef_df=pd.DataFrame({
    "feature":feature_cols,
    "coefficient":lr.coef_.round(4)
})
print(f'''
Coefficients (controlling for all other features):
{coef_df.sort_values("coefficient",ascending=False).to_string()}
''')


Coefficients (controlling for all other features):
          feature  coefficient
6        has_pool      12.1964
5         has_gym       7.9904
1        bedrooms       5.0556
7  locality_score       4.1055
2       bathrooms       3.2710
4           floor       0.3187
0       area_sqft       0.0401
3       age_years      -0.5148



### Regularisation comparison

In [8]:
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

ridge=Ridge(alpha=10.0)
lasso=Lasso(alpha=1.0)

ridge.fit(X_train_s,y_train)
lasso.fit(X_train_s,y_train)

print(f'''
Ridge RMSE: ₹{np.sqrt(mean_squared_error(y_test,ridge.predict(X_test_s))):.1f}L
Lasso RMSE: ₹{np.sqrt(mean_squared_error(y_test,lasso.predict(X_test_s))):.1f}L
Lasso zero coefficients: {np.sum(lasso.coef_==0).sum()}
''')


Ridge RMSE: ₹15.1L
Lasso RMSE: ₹15.5L
Lasso zero coefficients: 0



### Multicollinearity check

In [9]:
corr_matrix=X_train.corr()
high_corr=[
    (c1,c2,corr_matrix.loc[c1,c2])
    for c1 in feature_cols
    for c2 in feature_cols
    if c1<c2 and corr_matrix.loc[c1,c2]>0.7]
print(f'''
Highly correlated feature pairs (|r|>0.7): {high_corr if high_corr else 'None found'}
''')


Highly correlated feature pairs (|r|>0.7): None found



## LOGISTIC REGRESSION

In [10]:
X_cls,y_cls=make_classification(
    n_samples=3000,n_features=8,n_informative=5,
    n_redundant=2,class_sep=0.8,random_state=42
    )

feature_name=['rating','votes','avg_cost','age','delivary','parking','cuisine_score','locality_score']

X_train_c,X_test_c,y_train_c,y_test_c=train_test_split(X_cls,y_cls,test_size=0.2,random_state=42,stratify=y_cls)

scaler_c=StandardScaler()
X_train_cs=scaler_c.fit_transform(X_train_c)
X_test_cs=scaler.transform(X_test_c)

In [11]:
log_reg=LogisticRegression(max_iter=1000,random_state=42)
log_reg.fit(X_train_cs,y_train_c)

y_pred_c=log_reg.predict(X_test_cs)
y_prob_c=log_reg.predict_proba(X_test_cs)[:,1]

print(f'''
    {classification_report(y_test_c,y_pred_c)}
    AUC-ROC: {roc_auc_score(y_test_c,y_prob_c):.2f}
    Log Loss: {log_loss(y_test_c,y_pred_c)}

''')


                  precision    recall  f1-score   support

           0       0.59      0.52      0.56       300
           1       0.57      0.64      0.61       300

    accuracy                           0.58       600
   macro avg       0.58      0.58      0.58       600
weighted avg       0.58      0.58      0.58       600

    AUC-ROC: 0.65
    Log Loss: 15.018188912132148




### Odds ratio interpretation

In [12]:
odds_ratios=np.exp(log_reg.coef_[0])
print(f'''Odds Ratios (exp(coef)):''')
for name, OR in zip(feature_name,odds_ratios):
  direction='↑ increases' if OR > 1 else "↓ decreases"
  print(f'{name:20s}:{OR:.3f} ({direction} odds by {abs(OR-1)*100:.1f}%)')

Odds Ratios (exp(coef)):
rating              :0.986 (↓ decreases odds by 1.4%)
votes               :0.673 (↓ decreases odds by 32.7%)
avg_cost            :1.012 (↑ increases odds by 1.2%)
age                 :1.339 (↑ increases odds by 33.9%)
delivary            :1.576 (↑ increases odds by 57.6%)
parking             :1.189 (↑ increases odds by 18.9%)
cuisine_score       :1.261 (↑ increases odds by 26.1%)
locality_score      :0.477 (↓ decreases odds by 52.3%)


### Threshold tuning

In [13]:
threshold=np.arange(0.3,0.8,0.05)
print(f"'Threshold':>10 'Precision':>10 'Recall'>10 'F1':>10")
for thr in threshold:
  y_at_thr=(y_prob_c>=thr).astype(int)
  tp=((y_at_thr==1)&(y_test_c==1)).sum()
  fp=((y_at_thr==1)&(y_test_c==0)).sum()
  fn=((y_at_thr==0)&(y_test_c==0)==False).sum()
  prec=tp/(tp+fp) if (tp+fp)>0 else 0
  rec=tp/(sum(y_test_c)) if sum(y_test_c)>0 else 0
  f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
print(f" {thr:>8.2f}    {prec:>9.3f}        {rec:>9.3f}     {f1:>9.3f}")

'Threshold':>10 'Precision':>10 'Recall'>10 'F1':>10
     0.75        0.619            0.373         0.466
